<a href="https://colab.research.google.com/github/Mykola-Rubliuk/Data-Analytic/blob/main/Restaurants.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ============================================================
# 1. Imports and settings
# ============================================================
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
)

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)
pd.set_option("display.width", 1000)

def evaluate(name, model):
  y_pred = model.predict(X_test)
  y_prob = model.predict_proba(X_test)[:, 1]  # probability of class 1
  print(f"\n=== {name} ===")
  print(f"Accuracy:  {accuracy_score(y_test, y_pred):.3f}")
  print(f"Precision: {precision_score(y_test, y_pred):.3f}")
  print(f"Recall:    {recall_score(y_test, y_pred):.3f}")
  print(f"F1-score:  {f1_score(y_test, y_pred):.3f}")
  print(f"ROC-AUC:   {roc_auc_score(y_test, y_prob):.3f}")


# ============================================================
# 2. Data loading
# ============================================================
dictionary_df = pd.read_csv("data_dictionary_restaurant.csv")
restaurants_df = pd.read_csv("restaurant_survival.csv")

# print(dictionary_df)  # data dictionary: useful for checking leakage


# ============================================================
# 3. Data understanding (EDA)
# ============================================================
print("Dataset shape:")
print(restaurants_df.shape)

restaurants_df.info()
print(restaurants_df.describe())

print("\nMissing values:")
print(restaurants_df.isna().sum().sort_values(ascending=False))

print("\nTarget distribution:")
print(restaurants_df["survived_3y"].value_counts(normalize=True))

# --- 3.1 Target distribution
sns.countplot(data=restaurants_df, x="survived_3y")
plt.title("Survival Distribution")
plt.xlabel("Survived")
plt.ylabel("Number of Restaurants")
plt.show()

# --- 3.2 Survival by marketing
sns.countplot(data=restaurants_df, x="has_marketing", hue="survived_3y")
plt.title("Survival by marketing")
plt.xlabel("Marketing")
plt.ylabel("Number of Restaurants")
plt.show()

# Assumes survived_3y is encoded as 0/1
survival_rate = restaurants_df.groupby("has_marketing")["survived_3y"].mean() * 100
print("\nSurvival rate (%) by marketing:")
print(survival_rate)


# ============================================================
# 4. Data preparation
# ============================================================
# Drop the target, the identifier, and columns that leak the outcome
X = restaurants_df.drop(
    columns=["survived_3y", "restaurant_id", "failure_reason", "years_survived"]
)
y = restaurants_df["survived_3y"]

# --- 4.1 Train / test split (stratified to keep class balance)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print("\nOriginal dataset:", len(X))
print("Training dataset:", len(X_train))
print("Testing dataset:", len(X_test))

# --- 4.2 Feature types
numeric_features = X.select_dtypes(include="number").columns
categorical_features = X.select_dtypes(exclude="number").columns

# --- 4.3 Preprocessing
# Numeric: fill missing values with the median, then standardize
numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

# Categorical: fill missing values with the most frequent category, then one-hot encode
categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore")),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)


# ============================================================
# 5. Modelling
# ============================================================
model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(max_iter=1000)),
    ]
)



model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]  # probability of class 1

print("Logistic Regression")
print("\nPredictions:", y_pred[:10])
print("Actual values:", y_test[:10].values)

model_rf = Pipeline([("preprocessor", preprocessor),
                     ("classifier", RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1))])
model_rf.fit(X_train, y_train)
y_pred_rf = model_rf.predict(X_test)
y_prob_rf = model_rf.predict_proba(X_test)[:, 1]  # probability of class 1

print("\nRandom Forest")
print("\nPredictions:", y_pred_rf[:10])
print("Actual values:", y_test[:10].values)
# ============================================================
# 6. Evaluation
# ============================================================
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_prob)  # uses probabilities, not hard predictions

print("Logistic Regression")
print(f"\nAccuracy:  {accuracy * 100:.2f} %")
print(f"Precision: {precision * 100:.2f} %")
print(f"Recall:    {recall * 100:.2f} %")
print(f"F1-score:  {f1 * 100:.2f} %")
print(f"ROC-AUC:   {roc_auc * 100:.2f} %")

print("\nRandom Forest")
print(f"Accuracy:  {accuracy_score(y_test, y_pred_rf) * 100:.2f} %")
print(f"Precision: {precision_score(y_test, y_pred_rf) * 100:.2f} %")
print(f"F1: {f1_score(y_test, y_pred_rf) * 100:.2f} %")
print(f"ROC-AUC: {roc_auc_score(y_test, y_prob_rf) * 100:.2f}  %")

print("\nClassification report:")
print(classification_report(y_test, y_pred))

# --- 6.1 Confusion matrix
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

# --- 6.2 Metrics summary table
results_df = pd.DataFrame(
    {
        "Metric": ["Accuracy", "Precision", "Recall", "F1-score", "ROC-AUC"],
        "Score": [accuracy, precision, recall, f1, roc_auc],
    }
)
print("\n", results_df)


# ============================================================
# 7. Saving the model
# ============================================================
joblib.dump(model, "restaurants_model.pkl")
print("\nModel saved to restaurants_model.pkl")


FileNotFoundError: [Errno 2] No such file or directory: 'data_dictionary_restaurant.csv'